In [28]:
import pandas as pd

df = pd.read_csv('../data/archive/Disease_symptom_and_patient_profile_dataset.csv')
print(df.shape)      # how many rows and columns
print(df.columns)    # what the column names are
print(df.head())     # first 5 rows

(349, 10)
Index(['Disease', 'Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Age',
       'Gender', 'Blood Pressure', 'Cholesterol Level', 'Outcome Variable'],
      dtype='str')
       Disease Fever Cough Fatigue Difficulty Breathing  Age  Gender  \
0    Influenza   Yes    No     Yes                  Yes   19  Female   
1  Common Cold    No   Yes     Yes                   No   25  Female   
2       Eczema    No   Yes     Yes                   No   25  Female   
3       Asthma   Yes   Yes      No                  Yes   25    Male   
4       Asthma   Yes   Yes      No                  Yes   25    Male   

  Blood Pressure Cholesterol Level Outcome Variable  
0            Low            Normal         Positive  
1         Normal            Normal         Negative  
2         Normal            Normal         Negative  
3         Normal            Normal         Positive  
4         Normal            Normal         Positive  


In [29]:
import sqlite3

# This creates symptoms.db inside your data folder
# If it already exists it just connects to it
conn = sqlite3.connect('../data/symptoms.db')
cursor = conn.cursor()

In [12]:
# Create the diseases table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS diseases (
        disease_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL
    )
""")

# Create the symptoms table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS symptoms (
        symptom_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL
    )
""")

# Create the linking table that connects diseases to symptoms
cursor.execute("""
    CREATE TABLE IF NOT EXISTS disease_symptoms (
        disease_id INTEGER,
        symptom_id INTEGER,
        FOREIGN KEY (disease_id) REFERENCES diseases(disease_id),
        FOREIGN KEY (symptom_id) REFERENCES symptoms(symptom_id)
    )
""")

conn.commit()
print("Tables created successfully")

Tables created successfully


In [17]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS patient_profiles (
        profile_id INTEGER PRIMARY KEY AUTOINCREMENT,
        disease_id INTEGER,
        age INTEGER,
        gender TEXT,
        blood_pressure TEXT,
        cholesterol_level TEXT,
        outcome TEXT,
        FOREIGN KEY (disease_id) REFERENCES diseases(disease_id)
    )
""")
conn.commit()
print("Created patient_profiles table")

Created patient_profiles table


In [13]:
# First get all unique diseases and insert them, and only keep unique copies each run

unique_diseases = df['Disease'].unique()

print(unique_diseases)

for disease in unique_diseases:
    cursor.execute("""
        INSERT OR IGNORE INTO diseases (name) VALUES (?)
    """, (disease,))

for disease in unique_diseases:
    cursor.execute("""
        DELETE FROM diseases
            WHERE disease_id NOT IN (
                SELECT MIN(disease_id)
                FROM diseases
                GROUP BY name
        )
    """)

conn.commit()
print(f"Inserted {len(unique_diseases)} diseases")

<ArrowStringArray>
[        'Influenza',       'Common Cold',            'Eczema',
            'Asthma',   'Hyperthyroidism', 'Allergic Rhinitis',
 'Anxiety Disorders',          'Diabetes',   'Gastroenteritis',
      'Pancreatitis',
 ...
          'Dementia',    'Diverticulitis',             'Mumps',
     'Cholecystitis',   'Prostate Cancer',     'Schizophrenia',
              'Gout', 'Testicular Cancer',       'Tonsillitis',
 'Williams Syndrome']
Length: 116, dtype: str
Inserted 116 diseases


In [16]:
# Then get all unique symptoms and insert them
# Your CSV might have columns like Symptom1, Symptom2, Symptom3

# Still have to deal w the patient profile things like age, gender, etc. these are yes/no symptoms.

symptom_columns = ['Fever', 'Cough', 'Fatigue', 'Difficulty Breathing']

for symptom in symptom_columns:
    cursor.execute("""
        INSERT INTO symptoms (name) VALUES (?)
    """, (symptom,))

for symptom in symptom_columns:
    cursor.execute("""
        DELETE FROM symptoms
            WHERE symptom_id NOT IN (
                SELECT MIN(symptom_id)
                FROM symptoms
                GROUP BY name
        )
    """)

conn.commit()
print(f"Inserted {len(symptom_columns)} symptoms")

Inserted 4 symptoms


In [18]:
# Then populate the linking table. links disease id's to the symptoms that are yes's for that disease.

for _, row in df.iterrows():
    # Get disease_id
    cursor.execute("SELECT disease_id FROM diseases WHERE name = ?", 
                   (row['Disease'],))
    result = cursor.fetchone()
    if not result:
        continue
    disease_id = result[0]

    # For each symptom column, if value is Yes, link it
    for symptom_name in symptom_columns:
        if row[symptom_name] == 'Yes':
            cursor.execute("SELECT symptom_id FROM symptoms WHERE name = ?", 
                           (symptom_name,))
            sym_result = cursor.fetchone()
            if sym_result:
                cursor.execute("""
                    INSERT INTO disease_symptoms (disease_id, symptom_id)
                    VALUES (?, ?)
                """, (disease_id, sym_result[0]))

conn.commit()
print("Linking table populated")

Linking table populated


In [19]:
# Fill in the patient profile table. 

for _, row in df.iterrows():
    cursor.execute("SELECT disease_id FROM diseases WHERE name = ?", 
                   (row['Disease'],))
    result = cursor.fetchone()
    if not result:
        continue
    disease_id = result[0]

    cursor.execute("""
        INSERT INTO patient_profiles 
        (disease_id, age, gender, blood_pressure, cholesterol_level, outcome)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (disease_id, row['Age'], row['Gender'], 
          row['Blood Pressure'], row['Cholesterol Level'], 
          row['Outcome Variable']))

conn.commit()
print("Patient profiles populated")

Patient profiles populated


In [30]:
print("\n--- Verification ---")

cursor.execute("SELECT COUNT(*) FROM symptoms")
print("Symptoms:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM disease_symptoms")
print("Disease-symptom links:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM patient_profiles")
print("Patient profiles:", cursor.fetchone()[0])

# Sample joined query
result = pd.read_sql_query("""
    SELECT d.name as disease, s.name as symptom
    FROM diseases d
    JOIN disease_symptoms ds ON d.disease_id = ds.disease_id
    JOIN symptoms s ON s.symptom_id = ds.symptom_id
    LIMIT 30
""", conn)
print("\nSample disease-symptom links:")
print(result)


--- Verification ---
Symptoms: 4
Disease-symptom links: 672
Patient profiles: 349

Sample disease-symptom links:
              disease               symptom
0           Influenza                 Fever
1           Influenza               Fatigue
2           Influenza  Difficulty Breathing
3         Common Cold                 Cough
4         Common Cold               Fatigue
5              Eczema                 Cough
6              Eczema               Fatigue
7              Asthma                 Fever
8              Asthma                 Cough
9              Asthma  Difficulty Breathing
10             Asthma                 Fever
11             Asthma                 Cough
12             Asthma  Difficulty Breathing
13             Eczema                 Fever
14          Influenza                 Fever
15          Influenza                 Cough
16          Influenza               Fatigue
17          Influenza  Difficulty Breathing
18          Influenza                 Fever
19    

In [ ]:
conn.close()